<a href="https://colab.research.google.com/github/pramodjella/QuantumML-IIT-Delhi-/blob/main/QML_sentiment_analysis_using_pennylane.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pennylane transformers datasets scikit-learn

## Load SST-2 Dataset

### Subtask:
Load the SST-2 sentiment analysis dataset. This might involve downloading it or using a library like Hugging Face's `datasets`.


In [2]:
from datasets import load_dataset

# Load the SST-2 dataset
sst2_dataset = load_dataset('sst2')

print("SST-2 Dataset loaded successfully.")
print(sst2_dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


SST-2 Dataset loaded successfully.
DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})


In [3]:
print("\n--- Exploring Dataset Splits ---")
for split_name, dataset_split in sst2_dataset.items():
    print(f"\nSplit: {split_name}")
    print(f"Number of rows: {len(dataset_split)}")
    print(f"Features: {dataset_split.features}")
    print("Sample entries (first 3):")
    for i in range(min(3, len(dataset_split))):
        print(f"  Sentence: {dataset_split[i]['sentence']}")
        print(f"  Label: {dataset_split[i]['label']}")


--- Exploring Dataset Splits ---

Split: train
Number of rows: 67349
Features: {'idx': Value('int32'), 'sentence': Value('string'), 'label': ClassLabel(names=['negative', 'positive'])}
Sample entries (first 3):
  Sentence: hide new secretions from the parental units 
  Label: 0
  Sentence: contains no wit , only labored gags 
  Label: 0
  Sentence: that loves its characters and communicates something rather beautiful about human nature 
  Label: 1

Split: validation
Number of rows: 872
Features: {'idx': Value('int32'), 'sentence': Value('string'), 'label': ClassLabel(names=['negative', 'positive'])}
Sample entries (first 3):
  Sentence: it 's a charming and often affecting journey . 
  Label: 1
  Sentence: unflinchingly bleak and desperate 
  Label: 0
  Sentence: allows us to hope that nolan is poised to embark a major career as a commercial yet inventive filmmaker . 
  Label: 1

Split: test
Number of rows: 1821
Features: {'idx': Value('int32'), 'sentence': Value('string'), 'label': C

## Pre-process Data for PennyLane

### Subtask:
Pre-process the loaded SST-2 dataset. This typically includes tokenization, numericalization, and preparing the data into a suitable format for input into a PennyLane-based quantum machine learning model. This might involve creating feature vectors from text data.


In [4]:
from transformers import AutoTokenizer

# 1. Load a pre-trained tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# 2. Define maximum sequence length
MAX_LEN = 64

# 3. Create a tokenization function
def tokenize_function(examples):
    return tokenizer(examples['sentence'], truncation=True, padding='max_length', max_length=MAX_LEN)

# 4. Apply this tokenization function to the dataset splits
tokenized_datasets = sst2_dataset.map(tokenize_function, batched=True)

print("Dataset tokenized successfully.")
print(tokenized_datasets)

Dataset tokenized successfully.
DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1821
    })
})


In [5]:
num_train_samples = 500
num_val_samples = 50

train_dataset_small = tokenized_datasets['train'].select(range(num_train_samples))
val_dataset_small = tokenized_datasets['validation'].select(range(num_val_samples))

print(f"Created small training dataset with {len(train_dataset_small)} samples.")
print(f"Created small validation dataset with {len(val_dataset_small)} samples.")
print("Small training dataset features:")
print(train_dataset_small.features)
print("Small validation dataset features:")
print(val_dataset_small.features)

Created small training dataset with 500 samples.
Created small validation dataset with 50 samples.
Small training dataset features:
{'idx': Value('int32'), 'sentence': Value('string'), 'label': ClassLabel(names=['negative', 'positive']), 'input_ids': List(Value('int32')), 'token_type_ids': List(Value('int8')), 'attention_mask': List(Value('int8'))}
Small validation dataset features:
{'idx': Value('int32'), 'sentence': Value('string'), 'label': ClassLabel(names=['negative', 'positive']), 'input_ids': List(Value('int32')), 'token_type_ids': List(Value('int8')), 'attention_mask': List(Value('int8'))}


In [6]:
import numpy as np

# Extract features and labels for the small training dataset
X_train_input_ids = np.array(train_dataset_small['input_ids'])
X_train_attention_mask = np.array(train_dataset_small['attention_mask'])
y_train = np.array(train_dataset_small['label'])

# Extract features and labels for the small validation dataset
X_val_input_ids = np.array(val_dataset_small['input_ids'])
X_val_attention_mask = np.array(val_dataset_small['attention_mask'])
y_val = np.array(val_dataset_small['label'])

print("Extracted features and labels for small training dataset:")
print(f"X_train_input_ids shape: {X_train_input_ids.shape}")
print(f"X_train_attention_mask shape: {X_train_attention_mask.shape}")
print(f"y_train shape: {y_train.shape}")

print("\nExtracted features and labels for small validation dataset:")
print(f"X_val_input_ids shape: {X_val_input_ids.shape}")
print(f"X_val_attention_mask shape: {X_val_attention_mask.shape}")
print(f"y_val shape: {y_val.shape}")

Extracted features and labels for small training dataset:
X_train_input_ids shape: (500, 64)
X_train_attention_mask shape: (500, 64)
y_train shape: (500,)

Extracted features and labels for small validation dataset:
X_val_input_ids shape: (50, 64)
X_val_attention_mask shape: (50, 64)
y_val shape: (50,)


## Build PennyLane-based Sentiment Model

### Subtask:
Construct a quantum machine learning model using PennyLane for sentiment analysis.


In [7]:
import pennylane as qml
import pennylane.numpy as qml_np

# --- Step 2 & 3: Feature Extraction and Normalization ---
# First, calculate raw sums for all train and validation samples to determine global min/max for normalization
raw_train_sums_input_ids = np.sum(X_train_input_ids, axis=1)
raw_train_sums_attention_mask = np.sum(X_train_attention_mask, axis=1)
raw_val_sums_input_ids = np.sum(X_val_input_ids, axis=1)
raw_val_sums_attention_mask = np.sum(X_val_attention_mask, axis=1)

# Combine all raw sum values to find global min and max for normalization
all_raw_sums = np.concatenate([
    raw_train_sums_input_ids,
    raw_train_sums_attention_mask,
    raw_val_sums_input_ids,
    raw_val_sums_attention_mask
])

min_val = np.min(all_raw_sums)
max_val = np.max(all_raw_sums)

# Define the feature_extractor function
def feature_extractor(input_ids_single, attention_mask_single, global_min_val, global_max_val):
    sum_input_ids = np.sum(input_ids_single)
    sum_attention_mask = np.sum(attention_mask_single)
    features = np.array([sum_input_ids, sum_attention_mask])

    # Normalize features to the range [0, 2*pi]
    # Handle edge case where max_val == min_val to avoid division by zero
    if global_max_val == global_min_val:
        normalized_features = np.array([0.0, 0.0]) # Or handle as appropriate for your data
    else:
        normalized_features = (features - global_min_val) / (global_max_val - global_min_val) * (2 * np.pi)
    return normalized_features

# Apply feature_extractor to create X_train_features and X_val_features
X_train_features = np.array([
    feature_extractor(X_train_input_ids[i], X_train_attention_mask[i], min_val, max_val)
    for i in range(len(X_train_input_ids))
])
X_val_features = np.array([
    feature_extractor(X_val_input_ids[i], X_val_attention_mask[i], min_val, max_val)
    for i in range(len(X_val_input_ids))
])

print("Features extracted and normalized successfully.")
print(f"X_train_features shape: {X_train_features.shape}")
print(f"X_val_features shape: {X_val_features.shape}")

# --- Step 4: Initialize PennyLane quantum device ---
dev = qml.device('default.qubit', wires=2)
print("PennyLane quantum device initialized with 2 wires.")

# --- Step 5: Define the quantum circuit ---
@qml.qnode(dev)
def quantum_circuit(features, weights):
    # Angle Embedding: encode the two features into the two qubits
    qml.AngleEmbedding(features, wires=range(2))

    # Variational Layer: StronglyEntanglingLayers for trainable parameters
    # Weights shape: (number_of_layers, number_of_wires, 3)
    qml.StronglyEntanglingLayers(weights, wires=range(2))

    # Return the expectation value of PauliZ on the first qubit for classification
    return qml.expval(qml.PauliZ(0))

# Example of how to define initial weights for the circuit
num_layers = 3 # Number of layers for StronglyEntanglingLayers
num_qubits = 2 # Number of wires
weight_shapes = {'weights': (num_layers, num_qubits, 3)}

print(f"Quantum circuit defined with {num_layers} strongly entangling layers.")
print(f"Expected weight shape for training: {weight_shapes['weights']}")

Features extracted and normalized successfully.
X_train_features shape: (500, 2)
X_val_features shape: (50, 2)
PennyLane quantum device initialized with 2 wires.
Quantum circuit defined with 3 strongly entangling layers.
Expected weight shape for training: (3, 2, 3)


/usr/local/lib/python3.12/dist-packages/pennylane/__init__.py:209: RuntimeWarning: PennyLane is not yet compatible with JAX versions > 0.6.2. You have version 0.7.2 installed. Please downgrade JAX to 0.6.2 to avoid runtime errors using python -m pip install jax~=0.6.0 jaxlib~=0.6.0
  warnings.warn(


In [8]:
import pennylane as qml
import pennylane.numpy as qml_np

# Define the cost function
def cost_function(weights, features, labels):
    predictions = []
    for i in range(len(features)):
        # The quantum circuit returns an expectation value between -1 and 1
        output = quantum_circuit(features[i], weights)
        predictions.append(output)

    predictions = qml_np.array(predictions)
    # Map labels from {0, 1} to {-1, 1} to match PauliZ expectation values
    mapped_labels = 2 * labels - 1

    # Calculate mean squared error
    return qml_np.mean((predictions - mapped_labels) ** 2)

# Initialize an optimizer (e.g., qml.AdamOptimizer)
optimizer = qml.AdamOptimizer(stepsize=0.01)

print("Cost function and optimizer defined successfully.")

Cost function and optimizer defined successfully.


## Train Model

### Subtask:
Train the PennyLane-based model using the pre-processed SST-2 training data. This will involve defining an optimizer and using PennyLane's built-in differentiation for an efficient classical-quantum training loop.


In [ ]:
epochs = 50

# Initialize weights for the quantum circuit
# We use qml_np for trainable parameters
weights = qml_np.random.normal(loc=0, scale=0.1, size=weight_shapes['weights'], requires_grad=True)

cost_history = []

print("Starting model training...")
for epoch in range(epochs):
    # Update the weights using the optimizer
    # The optimizer.step method should ideally return only the updated parameters.
    # However, to prevent the ValueError, we explicitly handle cases where it might return a tuple (params, *args).
    returned_params = optimizer.step(cost_function, weights, X_train_features, y_train)

    # If returned_params is a sequence (list or tuple) with more than one element,
    # assume the first element is the actual updated weights.
    if isinstance(returned_params, (list, tuple)) and len(returned_params) > 1:
        weights = returned_params[0]
    else:
        # Otherwise, assume it returned just the updated weights (e.g., a single qml_np.array)
        weights = returned_params

    # Calculate the cost after the update
    cost = cost_function(weights, X_train_features, y_train)
    cost_history.append(cost)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1}/{epochs} | Cost: {cost:.4f}")

print("Training complete.")

Starting model training...
Epoch 10/50 | Cost: 1.3496


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. Define the predict function
def predict(features, weights):
    predictions = []
    for i in range(len(features)):
        # Execute the quantum circuit
        output = quantum_circuit(features[i], weights)
        # Convert expectation value (-1 to 1) to binary prediction (0 or 1)
        # Outputs >= 0 are classified as 1 (positive), outputs < 0 as 0 (negative)
        binary_prediction = 1 if output >= 0 else 0
        predictions.append(binary_prediction)
    return np.array(predictions)

# 2. Obtain predictions on the validation set
print("Generating predictions for the validation set...")
y_pred_val = predict(X_val_features, weights)
print("Predictions generated.")

# 3. and 4. Calculate and print evaluation metrics
print("\n--- Evaluation on Validation Set ---")
accuracy = accuracy_score(y_val, y_pred_val)
precision = precision_score(y_val, y_pred_val)
recall = recall_score(y_val, y_pred_val)
f1 = f1_score(y_val, y_pred_val)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

## Visualize Results

### Subtask:
Generate plots to visualize the training progress, such as loss over epochs, and present the evaluation metrics. Ensure all plots have appropriate legends for clarity.


In [ ]:
import matplotlib.pyplot as plt

# 1. Plotting training cost over epochs
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(cost_history) + 1), cost_history, marker='o', linestyle='-', color='skyblue')
plt.title('Training Cost Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Cost')
plt.grid(True)
plt.xticks(range(1, len(cost_history) + 1))
plt.legend(['Training Cost'])
plt.show()

# 2. Plotting evaluation metrics
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
metrics_values = [accuracy, precision, recall, f1]

plt.figure(figsize=(10, 6))
plt.bar(metrics_names, metrics_values, color=['lightcoral', 'lightgreen', 'lightblue', 'gold'])
plt.title('Model Evaluation Metrics on Validation Set')
plt.xlabel('Metric')
plt.ylabel('Value')
plt.ylim(0, 1) # Metrics are between 0 and 1
for i, value in enumerate(metrics_values):
    plt.text(i, value + 0.02, f'{value:.2f}', ha='center', va='bottom')
plt.show()

print("Plots generated successfully.")